# 10주차 실습 — 안전한 데이터 수정과 트랜잭션

- **데이터**: `studies` 에서 5행을 떼어 만든 연습용 표 `연습_시험`. 원본은 그대로 둡니다.
- **오늘 배우는 것**: ① 추가 · 수정 · 삭제의 기능을 알아본다 ② WHERE가 바꿀 대상을 정함을 안다 ③ 트랜잭션과 확정 · 되돌림을 이해한다 ④ 원본과 연습용 사본을 구분한다
- **점수는 없습니다.**

## 0. 시작하기 전에

- Colab 에서는 먼저 **`File → Save a copy in Drive`** 를 누르세요.
- 셀은 **위에서부터 차례로** 실행합니다 (▶ 또는 `Shift + Enter`). `%%sql` 아래의 `-- Q1` · `-- S1` 같은 문항 이름 줄은 지우지 마세요.
- 문항 SQL 을 실행하면 결과 아래에 **✅ / ❌** 가 바로 나옵니다.
- 오류가 나면 **`… Error:` 로 시작하는 줄**만 읽으면 됩니다.

In [ ]:
#@title [1번] ▶ 준비 셀 — 실행만 하세요 (Shift+Enter). 코드는 몰라도 됩니다. { display-mode: "form" }
import os, sys, urllib.request
import numpy as _np, pandas as _pd
DB  = "aact_mini.duckdb"
URL = "https://raw.githubusercontent.com/kevinkwshin/medical-db-course-2026/main/data/aact_mini.duckdb.gz"

# 로컬(수업 폴더)에서 열었을 때는 ../data/ 의 파일을 사용
for cand in [DB, os.path.join("..", "data", DB), os.path.join("data", DB)]:
    if os.path.exists(cand):
        DB = cand
        break
else:
    print("데이터 파일을 내려받습니다 (약 50MB, 1분 내외) ...")
    import gzip, shutil
    urllib.request.urlretrieve(URL, DB + ".gz")
    with gzip.open(DB + ".gz", "rb") as f_in, open(DB, "wb") as f_out:
        shutil.copyfileobj(f_in, f_out)
    os.remove(DB + ".gz")

if "google.colab" in sys.modules:           # Colab 에서만 설치 (수업에서 검증한 버전)
    import subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "duckdb==1.4.5", "jupysql==0.11.1", "duckdb-engine==0.17.0"])

import duckdb
_pd.set_option("display.max_colwidth", 60)
_pd.set_option("display.max_columns", None)     # SELECT * 로 열 이름을 볼 때 모든 열을 보여 줌
%load_ext sql
%config SqlMagic.autopandas = True
%config SqlMagic.displaylimit = 30
%config SqlMagic.feedback = 0
%xmode Minimal
try:                                         # jupysql 의 영어 안내문(If using snippets ...)을 숨김
    import sql.error_handler as _eh
    _eh.CTE_MSG, _eh.ORIGINAL_ERROR = "↓ 아래 '… Error:' 로 시작하는 줄이 원인입니다", ""
except Exception:
    pass
conn = duckdb.connect(DB, read_only=True)
conn.execute("SET python_enable_replacements = false")   # 빈칸 ___ 을 파이썬 변수로 읽지 않게

class _SameCursor:                           # jupysql 이 셀마다 새 cursor(=새 연결)를 열면
    def __init__(self, c): self._c = c       # CREATE TEMP TABLE 로 만든 표가 다음 셀에서 사라지므로
    def close(self): pass                    # 항상 같은 연결을 쓰게 합니다
    def __getattr__(self, k): return getattr(self._c, k)

class _OneConnection:
    def __init__(self, c): self._c = c
    def cursor(self): return _SameCursor(self._c)
    def __getattr__(self, k): return getattr(self._c, k)

_aact = _OneConnection(conn)
%sql _aact --alias aact

import hashlib as _hl, math as _math, decimal as _dec, datetime as _dt
import re as _re, zlib as _zlib, base64 as _b64, json as _json

def _load_answers(blob):
    """자동 확인용 지문(정답 결과표의 해시 · 행 수 · 열 이름 · 힌트)을 풉니다. 정답 SQL 은 들어 있지 않습니다."""
    return _json.loads(_zlib.decompress(_b64.b64decode(blob)).decode())

def _v(v):
    """값 하나를 비교용 문자열로: 빈 값 → '', 1947.0 → '1947', 날짜 → 'YYYY-MM-DD'"""
    try:
        if v is None or (not isinstance(v, (str, bytes)) and _pd.isna(v)):
            return ""
    except (TypeError, ValueError):
        pass
    if isinstance(v, _dec.Decimal):
        v = float(v)
    if isinstance(v, (bool, _np.bool_)):
        return str(bool(v))
    if isinstance(v, (int, _np.integer)):
        return str(int(v))
    if isinstance(v, (float, _np.floating)):
        v = round(float(v), 4)
        return str(int(v)) if v.is_integer() else repr(v)
    if isinstance(v, (_pd.Timestamp, _dt.datetime)):
        if v.hour == 0 and v.minute == 0 and v.second == 0 and v.microsecond == 0:
            return v.strftime("%Y-%m-%d")
        return v.isoformat()
    if isinstance(v, _dt.date):
        return v.isoformat()
    return str(v)

def _sigs(df):
    """결과표의 지문: 열 이름(소문자), 행 수, 행 순서 무시 지문(sig), 행 순서 포함 지문(osig)"""
    cols = [str(c).strip().lower() for c in df.columns]
    rows = ["\x1f".join(_v(x) for x in r) for r in df.itertuples(index=False, name=None)]
    head = "\x1e".join(cols)
    h = lambda rs: _hl.md5((head + "\x1d" + "\x1e".join(rs)).encode()).hexdigest()[:12]
    return {"sig": h(sorted(rows)), "osig": h(rows), "rows": len(rows), "cols": cols}

_LAST = {"tag": None, "error": False, "sql": ""}
_TRIES, _RESULT = {}, {}

def _after_cell(result):
    info = getattr(result, "info", None)
    src = (getattr(info, "raw_cell", "") or "").lstrip()
    if not src.startswith("%%sql"):
        return
    tag = None
    for line in src.split("\n")[1:]:
        s = line.strip()
        if s.startswith("--"):
            parts = s[2:].split()
            if parts and _re.match(r"^\[\d+번\]$", parts[0]):      # 수업용 셀 번호 [12번] 은 건너뜀
                parts = parts[1:]
            tag = parts[0] if parts and not parts[0].startswith("[") else None
            break
        if s:
            break
    body = "\n".join(l.split("--")[0] for l in src.split("\n")[1:])
    _LAST.update(tag=tag, error=not result.success, sql=body)
    if tag and tag in globals().get("ANSWERS", {}):         # 문항 셀(-- Q1)이면 실행하자마자 ✅/❌ 표시
        check(tag, getattr(result, "result", None))

for _cb in list(get_ipython().events.callbacks["post_run_cell"]):
    if getattr(_cb, "__name__", "") == "_after_cell":
        get_ipython().events.unregister("post_run_cell", _cb)
get_ipython().events.register("post_run_cell", _after_cell)

def _expected(name):
    a = ANSWERS[name]
    if "by_mesh" in a:
        if not globals().get("MY_MESH"):
            return None
        return dict(a["by_mesh"][MY_MESH], hint=a.get("hint", ""), ordered=a.get("ordered", False))
    return a

def _miss(name, why, exp):
    """틀렸을 때: 기록하고, 두 번째부터 힌트를 붙여 출력"""
    _RESULT[name] = False
    _TRIES[name] = _TRIES.get(name, 0) + 1
    msg = f"❌ {name}  {why}"
    hint = (exp or ANSWERS.get(name, {})).get("hint")
    if hint:
        msg += ("\n   💡 힌트: " + hint) if _TRIES[name] >= 2 else "\n   (한 번 더 다르면 힌트가 나옵니다)"
    print(msg)

def check(name, df=None):
    """바로 위 %%sql 셀의 결과를 확인합니다.  예) check("Q1", _)"""
    if name not in ANSWERS:
        print(f"알 수 없는 문항 이름입니다: {name}"); return
    if _LAST["tag"] and _LAST["tag"] != name:
        print(f"⚠️ 방금 실행한 SQL 셀은 {_LAST['tag']} 입니다. {name} 의 %%sql 셀을 먼저 실행한 뒤 이 셀을 실행하세요."); return
    exp = _expected(name)
    if exp is None:
        print("⚠️ 분석 질환 정보가 없습니다. 맨 위 준비 셀을 다시 실행하세요."); return
    if _LAST["sql"] and not _re.search(r"\bFROM\b", _LAST["sql"], _re.I):
        return _miss(name, "FROM 이 없는 SQL 입니다. 숫자를 직접 적지 말고, 표에서 꺼내 세는 SQL 을 쓰세요.", exp)
    if _LAST["error"]:
        return _miss(name, "SQL 에서 오류가 났습니다. 붉은 글씨 중 'Error:' 가 들어간 줄을 읽고 고친 뒤 다시 실행하세요.", exp)
    if not isinstance(df, _pd.DataFrame):
        return _miss(name, "결과가 표가 아닙니다. SQL 이 SELECT 로 끝나는지 확인하세요.", exp)
    got = _sigs(df)
    if got["sig"] == exp["sig"] and (not exp.get("ordered") or got["osig"] == exp["osig"]):
        _RESULT[name] = True
        print(f"✅ {name} 정답!  (행 {got['rows']}개)"); return
    zero = "값의 철자 · 대소문자 · 따옴표(예: 'COMPLETED')와, 빈 값을 = 나 <> 로 비교하지 않았는지(IS NULL) 확인하세요."
    if got["sig"] == exp["sig"]:
        why = "값은 맞지만 행의 순서가 다릅니다. 정렬(ORDER BY · DESC)을 확인하세요."
    elif got["cols"] != exp["cols"]:
        if len(got["cols"]) != len(exp["cols"]):
            why = f"열 개수가 다릅니다 (기대 {len(exp['cols'])}개: {', '.join(exp['cols'])} / 현재 {len(got['cols'])}개)."
        elif sorted(got["cols"]) == sorted(exp["cols"]):
            why = f"열의 순서가 다릅니다. 문항에 적힌 순서대로 적으세요: {', '.join(exp['cols'])}"
        else:
            why = f"열 이름이 다릅니다. 기대: {', '.join(exp['cols'])}  /  현재: {', '.join(got['cols'])}  (AS 로 이름을 맞추세요)"
    elif got["rows"] == 0 and exp["rows"] > 0:
        why = "결과가 0행입니다. " + zero
    elif got["rows"] == 1 and len(df.columns) and all(_v(x) in ("0", "") for x in df.iloc[0]):
        why = "결과가 0 이거나 비어 있습니다. 조건에 맞는 행이 없다는 뜻입니다. " + zero
    elif got["rows"] != exp["rows"]:
        why = f"행 수가 다릅니다 (기대 {exp['rows']}행, 현재 {got['rows']}행). 조건과 LIMIT 을 확인하세요."
    elif exp.get("ordered"):
        why = "행 수는 맞지만 값이나 순서가 다릅니다. 조건과 정렬(ORDER BY · DESC)을 확인하세요."
    else:
        why = "행 수는 맞지만 값이 다릅니다. 조건이나 계산식을 확인하세요."
    _miss(name, why, exp)

def 진행확인():
    """지금까지 확인한 문항의 결과를 한눈에 봅니다. 선택 문항은 개수에서 뺍니다."""
    names = list(ANSWERS)
    core = [n for n in names if not ANSWERS[n].get("optional")]
    ok = [n for n in core if _RESULT.get(n) is True]
    print(f"정답 {len(ok)} / {len(core)} 문항")
    for n in names:
        mark = {True: "✅", False: "❌"}.get(_RESULT.get(n), "·  (아직 실행 안 함)")
        print(f"  {n:>4}  {mark}" + ("   (선택)" if ANSWERS[n].get("optional") else ""))

print("연결 완료:", DB)

ANSWERS = _load_answers("eNq9lV1PGkEUhv/KCTdCQgjflCa9sEqUNGKgGC+axvCxtCYUGqDthTFZKxqqNIWGlcUCwYQqWi+o0ooJvelP6eXO2f/QMysoYNukaZUbZt85c3bmfc6ZXdH5Lbq7sKJLLz+hf13IJghRISZYrWGbzgi65G/0VPJVmnQnDSPJOB8+0iUimaXlqLbqpZAKxeNL6Uwo8yLNFSGRSsbjz4RERveYHp8u04DSKidt5bQHakXCWhfUYhXYQQtQyrK6jOUC+KaCbreZ/yygSlVQd95grQPstIPlY5RopXTItnNsuwl6J00aTKC032NNBKWTVeUupcoBljuAuSpmqywvsr2qEZSuiPUC2yoB1osUzUptlDv0eiMFfupPKWfdi3gTUH7cprdLDWBSXlt3ImMtj+tr6nqVQvQGwA2RfRXVbBtwu0lbUSUZs13cLZk0R1JRISVE6cyxUDwtcOV5ZjmZCMUH0ippfuswC3PE5Xa5LPaIPeoeZjGu91lYhlnQdtWyjDn5F3bz/duAfMSdzoiHJlic9QQ8wJ2f4K7d425OADtq4U4Rqz06KWBjDbe+DBZQFHd7FLhxAGFi0RucnQ5MLvomQM/NPO6SucZx1w1/Y5FttFxd4Zjb6rLecZlHy3VU71vk+N/lSk5RWcHlMcFG5NlBHoKewJzXNxn0TJMXIrCCfN1sB8Vyb1mxAKxdYmIeyznu8mGnD4J97AE7b7Bqi+23QTlZx/rGYPVDT5CD0opxPccdN17h46rfClihRmhXOArcayufiVV2DOCtlbZ9pLTD0ZgtGg47XVHXSGmP6X1u1hvgBlY6iX7B98A3T9ycZjN8P4OAZyqw4A16fTNgMRs4SPa6RdfRGDnKoFE9z/6BWKM5gIsHIu52xvhdXTSXfUDkNvn1hqW1W8fjGMZjj9nc5og7Yo9FnMN4xvWbaitu7sU5tSbRX/UXMRogG8X1Q9w39G8o9i7PkVXEcWqUvyHpp+bn5rxBg2Y0fWhostaD+54Zr4/fieqHLNZk+sKIysk3rV3Km+ytTFNabvJWQ3a0wRrnQzSvc/I7AJsilci/81n9Cd8pg9I=")   # 자동 확인용 (정답 SQL 은 없음)

In [ ]:
#@title [2번] ▶ 수정 연습 준비 — 실행만 하세요 (Shift+Enter) { display-mode: "form" }
class _한연결:
    """%%sql 셀이 BEGIN 을 다음 셀의 COMMIT · ROLLBACK 까지 이어 가게 합니다."""
    def __init__(self, con): self.con = con
    def cursor(self): return self
    def commit(self): pass
    def close(self): pass
    def __getattr__(self, name): return getattr(self.con, name)

if "_rw" not in globals():
    _rw = _한연결(conn)
%sql _rw --alias rw

def 처음상태로():
    """연습_시험 을 처음 5행으로 되돌립니다."""
    try:
        _rw.execute("ROLLBACK")
    except Exception:
        pass
    _rw.execute("""CREATE OR REPLACE TEMP TABLE 연습_시험 AS
SELECT nct_id, overall_status, enrollment
FROM studies
ORDER BY nct_id LIMIT 5;""")
    print("연습_시험 을 처음 5행으로 되돌렸습니다.")

print("준비 완료")

## 1. INSERT · UPDATE · DELETE — 세 가지 문장

### 연습용 표 만들기 — studies 에서 5행만

**질문** 원본(studies)은 그대로 두고, 고치는 연습은 어디서 하나?

**원본** studies 는 그대로 (읽기 전용)

**사본** 연습_시험 은 노트북을 닫으면 사라짐. 아래 예시는 매번 이 5행에서 시작 (`처음상태로()`)

In [ ]:
%%sql
-- [3번] 연습용 표 만들기 → 5행
CREATE OR REPLACE TEMP TABLE 연습_시험 AS
SELECT nct_id, overall_status, enrollment
FROM studies
ORDER BY nct_id LIMIT 5;

SELECT * FROM 연습_시험 ORDER BY nct_id;

### INSERT · UPDATE · DELETE — 추가 · 수정 · 삭제

| 문장 | 하는 일 | 예 |
|---|---|---|
| INSERT | 새 행을 추가 | `INSERT INTO 연습_시험 (nct_id, overall_status, enrollment) VALUES ('NCT99999999', 'RECRUITING', 50);` |
| UPDATE | 기존 값을 바꿈 | `UPDATE 연습_시험 SET overall_status = 'COMPLETED' WHERE nct_id = 'NCT00623389';` |
| DELETE | 행을 지움 | `DELETE FROM 연습_시험 WHERE overall_status = 'WITHDRAWN';` |

WHERE는 수정 · 삭제할 대상을 정합니다. 없으면 모든 행이 대상입니다.

### INSERT — 새 행 추가하기

**질문** 연습_시험 에 새 시험 한 건을 넣으면?

**확인** 5행 → 6행. 맨 아래에 새 행이 생김

**VALUES** 열 순서(nct_id, overall_status, enrollment)대로 값을 적음

In [ ]:
처음상태로()   # [4번]

In [ ]:
%%sql
-- [5번] INSERT → 6행
INSERT INTO 연습_시험
  (nct_id, overall_status, enrollment)
VALUES ('NCT99999999', 'RECRUITING', 50);

SELECT * FROM 연습_시험 ORDER BY nct_id;

### UPDATE — 값 바꾸기

**질문** NCT00623389 시험의 상태만 COMPLETED 로 바꾸면?

**확인** WHERE 에 맞는 1행만 바뀜 (RECRUITING → COMPLETED)

**SET** 바꿀 열과 새 값, WHERE 는 바꿀 행

In [ ]:
처음상태로()   # [6번]

In [ ]:
%%sql
-- [7번] UPDATE → NCT00623389 만 COMPLETED
UPDATE 연습_시험
SET overall_status = 'COMPLETED'
WHERE nct_id = 'NCT00623389';

SELECT * FROM 연습_시험 ORDER BY nct_id;

### DELETE — 행 지우기

**질문** 철회된(WITHDRAWN) 시험을 지우면?

**확인** WITHDRAWN 3행이 지워지고 2행이 남음

**먼저 세기** 지우기 전에 같은 WHERE 로 SELECT COUNT(*) 를 하면 3

In [ ]:
처음상태로()   # [8번]

In [ ]:
%%sql
-- [9번] DELETE → 2행
DELETE FROM 연습_시험
WHERE overall_status = 'WITHDRAWN';

SELECT * FROM 연습_시험 ORDER BY nct_id;

### WHERE는 바꿀 대상을 정합니다 — 같은 조건으로 먼저 조회해 봅니다

| WHERE를 빠뜨리면 | 이렇게 확인하고 씁니다 |
|---|---|
| `UPDATE 연습_시험 SET overall_status = 'COMPLETED';` | `SELECT COUNT(*) FROM 연습_시험 WHERE overall_status = 'WITHDRAWN';` → 3 |
| 5개 시험이 모두 COMPLETED 로 바뀝니다 | `UPDATE 연습_시험 SET overall_status = 'TERMINATED' WHERE overall_status = 'WITHDRAWN';` → 조건에 맞는 3개 시험만 바뀝니다 |

In [ ]:
처음상태로()   # [10번]

In [ ]:
%%sql
-- [11번] WHERE 를 빠뜨림 → 5행 모두 COMPLETED
UPDATE 연습_시험
SET overall_status = 'COMPLETED';
-- WHERE 가 없습니다

SELECT * FROM 연습_시험 ORDER BY nct_id;

In [ ]:
처음상태로()   # [12번]

In [ ]:
%%sql
-- [13번] ① 같은 WHERE 로 먼저 세기 → 3
SELECT COUNT(*) FROM 연습_시험
WHERE overall_status = 'WITHDRAWN';

In [ ]:
%%sql
-- [14번] ② 같은 WHERE 로 수정 → 3행만 TERMINATED
UPDATE 연습_시험
SET overall_status = 'TERMINATED'
WHERE overall_status = 'WITHDRAWN';

SELECT * FROM 연습_시험 ORDER BY nct_id;

### 여기까지 정리

- ① INSERT는 추가, UPDATE는 변경, DELETE는 삭제
- ② WHERE는 수정 · 삭제할 대상을 정한다
- ③ 바꾸기 전에 같은 조건으로 SELECT 해 본다

**예시로 확인하기** 시험 한 건의 상태만 바꾸려면? → WHERE로 그 시험(nct_id)을 지정합니다.

## 2. 트랜잭션 — 한 덩어리로 묶기

**트랜잭션** = 함께 처리할 여러 변경을 하나로 묶는 것.

**입원 수속 (한 덩어리)**: ① 입원 기록을 만든다 → ② 병상을 ‘사용 중’으로 바꾼다 → ③ 수납 내역을 만든다

**②에서 오류가 나면?** 입원 기록은 있는데 병상은 비어 있는 상태. 데이터가 서로 맞지 않게 됩니다.

**그래서 묶습니다.** 하나라도 실패하면 전부 없던 일로 되돌립니다.

## 3. COMMIT과 ROLLBACK — 확정과 되돌리기

트랜잭션 안에서 바꾼 뒤, 결과를 보고 선택합니다.

| 명령 | 언제 | 하는 일 |
|---|---|---|
| `COMMIT;` | 결과가 맞으면 | 변경을 확정합니다 |
| `ROLLBACK;` | 이상하면 | 확정하지 않은 변경을 취소합니다 |

확정한 변경을 ROLLBACK으로 되돌릴 수는 없습니다.

### ROLLBACK 해 보기 — 실수를 되돌리기

**질문** BEGIN 뒤에 WHERE 없는 UPDATE 를 했다면?

**① UPDATE 뒤** WHERE 를 빠뜨려 5행 모두 COMPLETED

**② ROLLBACK 뒤** BEGIN 이후의 변경이 없던 일이 됨. COMMIT 한 뒤에는 되돌릴 수 없음

BEGIN 이 든 셀은 한 번만 실행하세요. 꼬이면 `처음상태로()` 를 실행합니다.

In [ ]:
처음상태로()   # [15번]

In [ ]:
%%sql
-- [16번] ① BEGIN 뒤 WHERE 없는 UPDATE → 모두 COMPLETED (아직 확정 전)
BEGIN;

UPDATE 연습_시험
SET overall_status = 'COMPLETED';   -- WHERE 없음

SELECT * FROM 연습_시험 ORDER BY nct_id;

In [ ]:
%%sql
-- [17번] ② ROLLBACK → 처음 5행대로
ROLLBACK;

SELECT * FROM 연습_시험 ORDER BY nct_id;

## 직접 해보기 — 5문항

각 문항은 `처음상태로()` 셀부터 실행하세요 (처음 5행에서 시작). 빈칸 `___` 을 채워 실행하면 결과 아래에 ✅ / ❌ 가 나옵니다.

**열 이름 찾기** — 표의 열 이름은 `DESCRIBE` 로 봅니다 (9주차 0단계와 같은 방법).

In [ ]:
%%sql
-- [18번] 연습_시험 의 열 이름 · 타입 보기
SELECT *
FROM (DESCRIBE 연습_시험);

**Q1.** 연습_시험 에 새 시험 한 행을 추가하세요. nct_id `'NCT99000001'`, overall_status `'RECRUITING'`, enrollment `40`.

In [ ]:
처음상태로()   # [19번]

In [ ]:
%%sql
-- [20번] Q1
INSERT INTO 연습_시험 (nct_id, overall_status, enrollment)
VALUES (___, ___, ___);

-- 결과 확인 (그대로 두세요)
SELECT * FROM 연습_시험 ORDER BY nct_id;

**Q2.** 철회된(`'WITHDRAWN'`) 시험이 몇 건인지 세세요.

쓰는 열: `overall_status`

In [ ]:
처음상태로()   # [21번]

In [ ]:
%%sql
-- [22번] 연습_시험 의 열 이름 · 타입 보기
SELECT *
FROM (DESCRIBE 연습_시험);

In [ ]:
%%sql
-- [23번] Q2
SELECT COUNT(*) AS 시험수
FROM 연습_시험
WHERE ___;

**Q3.** Q2 에서 센 WITHDRAWN 시험의 상태를 `'TERMINATED'` 로 바꾸세요. WHERE 는 **Q2 와 같게** 씁니다 (바꾸기 전에 먼저 센 것).

In [ ]:
처음상태로()   # [24번]

In [ ]:
%%sql
-- [25번] Q3
UPDATE 연습_시험
SET overall_status = ___
WHERE ___;

-- 결과 확인 (그대로 두세요)
SELECT * FROM 연습_시험 ORDER BY nct_id;

**Q4.** 등록 인원이 0명인 시험을 분석에서 빼려고 합니다. 등록 인원(`enrollment`)이 `0` 인 시험만 지우세요.

쓰는 열: `enrollment`

In [ ]:
처음상태로()   # [26번]

In [ ]:
%%sql
-- [27번] 연습_시험 의 열 이름 · 타입 보기
SELECT *
FROM (DESCRIBE 연습_시험);

In [ ]:
%%sql
-- [28번] Q4
DELETE FROM 연습_시험
WHERE ___;

-- 결과 확인 (그대로 두세요)
SELECT * FROM 연습_시험 ORDER BY nct_id;

**Q5.** 아래 준비 셀은 BEGIN 후 **WHERE 없는 UPDATE** 를 실행합니다 (5행 모두 COMPLETED, 아직 확정 전). `___` 에 명령을 넣어 되돌리세요.

In [ ]:
# [29번]
# Q5 준비: BEGIN 후 WHERE 없는 UPDATE
처음상태로()
%sql BEGIN;
%sql UPDATE 연습_시험 SET overall_status = 'COMPLETED';
%sql SELECT * FROM 연습_시험;

In [ ]:
%%sql
-- [30번] Q5
___;

-- 결과 확인 (그대로 두세요)
SELECT * FROM 연습_시험 ORDER BY nct_id;

## 오늘 배운 것

- 추가 · 수정 · 삭제 명령이 있다. WHERE로 대상을 정한다
- 여러 변경을 묶고, 확정하거나 확정 전에 되돌릴 수 있다
- 연습은 사본에서 하고, 원본은 그대로 둔다

다음 시간: 11강. CDW 구조와 분석 업무 흐름 — 병원에서 데이터를 받는 과정

In [ ]:
진행확인()   # [31번]